# Exploratory Data Analysis

In [120]:
# load packages
import pandas as pd
import numpy as np
import glob

In [121]:
# get list of file paths
file_paths = glob.glob("./Data/raw/citi_costco*.CSV")

# loop through each file in the folder and essentially union all the csvs together into one dataframe
citi_costco_transactions = pd.concat([pd.read_csv(file) for file in file_paths], ignore_index=True)

citi_costco_transactions.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 98 entries, 0 to 97
Data columns (total 6 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   Status       98 non-null     object 
 1   Date         98 non-null     object 
 2   Description  98 non-null     object 
 3   Debit        95 non-null     float64
 4   Credit       3 non-null      float64
 5   Member Name  96 non-null     object 
dtypes: float64(2), object(4)
memory usage: 4.7+ KB


### Extract
Lets figure out how to extract the data from the file

In [122]:
from dataclasses import dataclass
from pathlib import Path
from __future__ import annotations

@dataclass
class RawStatement:
    """Container for one raw CSV plus metadata about where it came from."""
    source_name: str          # e.g. "chase_credit_card" — matches config.yaml key
    file_path: Path
    df: pd.DataFrame

In [123]:
# Load the configuration
import yaml

def load_config(config_path: str = "./config.yaml") -> dict:
    """Load the source/column-mapping config."""
    # TODO: open config_path, yaml.safe_load(), return dict
    with open(config_path, "r") as f:
        config = yaml.safe_load(f)
    return config

config__ = load_config()
print(config__["sources"].items())

dict_items([('citi_costco_credit_card', {'file_pattern': 'citi_costco*.csv', 'column_map': {'month': None, 'date': 'Date', 'category': None, 'subcategory': None, 'amount': 'Debit', 'description': 'Description'}, 'amount_sign': 'positive_is_expense', 'date_format': '%m/%d/%Y'}), ('amex_gold_credit_card', {'file_pattern': 'amex_gold*.csv', 'column_map': {'month': None, 'date': 'Date', 'category': None, 'subcategory': None, 'amount': 'Debit', 'description': 'Description'}, 'amount_sign': 'positive_is_expense', 'date_format': '%m/%d/%Y'}), ('iq_credit_card', {'file_pattern': 'iq_credit_card*.csv', 'column_map': {'month': None, 'date': 'Date', 'category': None, 'subcategory': None, 'amount': 'Amount', 'description': 'Description'}, 'amount_sign': 'positive_is_expense', 'date_format': '%m/%d/%Y'}), ('iq_checking', {'file_pattern': 'iq_checking*.csv', 'column_map': {'month': None, 'date': 'Posting Date', 'category': None, 'subcategory': None, 'amount': 'Amount', 'description': 'Description'},

In [124]:
# this function will find all files in the raw_data_dir that match the file_pattern for a given source
def find_files_for_source(raw_data_dir: Path, file_pattern: str) -> list[Path]:
    """Glob raw_data_dir for files matching this source's pattern."""
    # TODO: return sorted list of Path objects matching file_pattern
    case_insensitive_pattern = file_pattern.replace(".csv", ".[cC][sS][vV]")
    file_paths = glob.glob(f"{raw_data_dir}/{case_insensitive_pattern}")
    return sorted(Path(f) for f in file_paths)

root_data_dir = "./Data/raw"
citi_files = find_files_for_source(root_data_dir, "citi_costco*.CSV")
amex_files = find_files_for_source(root_data_dir, "amex_gold_credit_card*.CSV")
iq_files = find_files_for_source(root_data_dir, "iq_credit_card*.CSV")

print("Citi files:", citi_files)
print("Amex files:", amex_files)
print("IQ files:", iq_files)


Citi files: [PosixPath('Data/raw/citi_costco_2026-06-19.CSV'), PosixPath('Data/raw/citi_costco_2026-07-21.CSV')]
Amex files: []
IQ files: []


In [125]:
def extract_all(config: dict) -> list[RawStatement]:
    """
    Main entry point for this module.
    Loop through every source in config['sources'], find matching files,
    read each into a DataFrame, and return a list of RawStatement objects.
    """
    raw_statements: list[RawStatement] = []
    root_data_dir = "./Data/raw"

    # TODO:
    for source_name, source_cfg in config["sources"].items():
        files = find_files_for_source(root_data_dir, source_cfg["file_pattern"])
        for f in files:
            df = pd.concat([pd.read_csv(f) for f in files], ignore_index=True)
            raw_statements.append(RawStatement(source_name, f, df))

    return raw_statements

if __name__ == "__main__":
    # Quick manual test when running this file directly
    cfg = load_config()
    statements = extract_all(cfg)
    print(f"Extracted { len(statements)} raw statement(s).")


Extracted 2 raw statement(s).


### Transform
Now lets transform the raw statements we pull from the extract portion. We'll want to normalize column names and data types across all sources.

    txn_id | month | date | category | subcategory | amount | description | source

`amount` convention: negative = money out (expense), positive = money in.
This is the "canonical" sign convention for the whole pipeline — resolve
each source's raw sign convention against this in normalize_amount_sign().

In [ ]:
# transform setup
from datetime import datetime
import hashlib

import pandas as pd

from extract import RawStatement # import RawStatement from extract.py

NORMALIZED_COLUMNS = ["txn_id", "month", "date", "category", "subcategory", "amount", "description", "source"]

In [ ]:
# rename columns
statements = extract_all(config__)

def rename_columns(df: pd.DataFrame, column_map: dict) -> pd.DataFrame:
    """
    Rename raw columns to normalized names using column_map from config.
    TODO: invert column_map (normalized -> raw) into (raw -> normalized)
          and call df.rename(columns=...). Handle missing/null mappings
          (e.g. category: null) by creating an empty column instead.
    """
    df = df.copy()
    for normalized_name, raw_name in column_map.items():
        if raw_name is None:
            df[normalized_name] = np.nan  # Create an empty column if mapping is None
        else:
            df = df.rename(columns={raw_name: normalized_name})
    return df

for statement in statements:
    source_name = statement.source_name
    df = statement.df
    column_map = config__["sources"][source_name]["column_map"]
    df_renamed = rename_columns(df, column_map)
    print(f"Renamed columns for {source_name}:")
    print(df_renamed.head())
